In [1]:
import numpy as np
from scipy.optimize import linprog
import pandas as pd
import matplotlib.pyplot as plt

Matplotlib is building the font cache; this may take a moment.


In [3]:

# Generate random 4*4 payoff matrices for two players 
U1 = np.random.uniform(0, 1, (4, 4))
U2 = np.random.uniform(0, 1, (4, 4))

print("Player 1 payoff matrix:")
print(U1)

print("Player 2 payoff matrix:")
print(U2)

Player 1 payoff matrix:
[[0.4026631  0.05455935 0.81751401 0.68802878]
 [0.20006758 0.57239761 0.59945755 0.03283138]
 [0.18932658 0.24150985 0.29850417 0.12456492]
 [0.91358603 0.79821753 0.22182845 0.86222389]]
Player 2 payoff matrix:
[[9.98677666e-01 3.42253632e-01 7.74760678e-01 6.58853641e-01]
 [3.69950154e-04 2.78161557e-01 2.21199472e-01 2.25161089e-01]
 [4.16016163e-01 7.92414018e-01 2.85452863e-01 4.96195548e-01]
 [2.61231554e-01 4.05646005e-01 1.90906374e-01 1.42680288e-01]]


In [4]:
# Example belief of Player 1 about Player 2's strategies
belief_p2 = np.array([0.25, 0.25, 0.25, 0.25])

# Expected payoff of each strategy for Player 1
expected_payoff_p1 = U1 @ belief_p2

print("Player 1 expected payoffs:")
print(expected_payoff_p1)

Player 1 expected payoffs:
[0.49069131 0.35118853 0.21347638 0.69896397]


In [ ]:
# Finding player 1's best response under one specific belief of m
strategies = ["A", "B", "C", "D"]

best_response_index = np.argmax(expected_payoff_p1)
best_response = strategies[best_response_index]

print("Player 1 best response:", best_response)

Player 1 best response: D


In [7]:
# Check whether strategy D can be a best response
# when Player 2 is believed to play only A or B

candidate_columns = [0, 1]   # A, B
target_strategy = 3          # D

A_eq = [[1, 1]]
b_eq = [1]

bounds = [(0, 1), (0, 1)]

A_ub = []
b_ub = []

for other_strategy in range(4): # Check all strategies for player 1
    if other_strategy == target_strategy: # we don't need to check for D
        continue

    difference = (
        U1[other_strategy, candidate_columns]
        - U1[target_strategy, candidate_columns]
    ) # check for wether the expected payoff of D is greater than the other strategies

    A_ub.append(difference)
    b_ub.append(0)

In [8]:
# Objective: we only care about feasibility
c = [0, 0]

result = linprog(
    c,
    A_ub=A_ub,
    b_ub=b_ub,
    A_eq=A_eq,
    b_eq=b_eq,
    bounds=bounds,
    method="highs"
)

print("Feasible:", result.success)

if result.success:
    print("A belief that makes D a best response:")
    print(result.x)

Feasible: True
A belief that makes D a best response:
[0. 1.]


In [9]:
strategies = ["A", "B", "C", "D"]

X1 = [0, 1, 2]   # A, B, C
X2 = [0, 1]      # A, B

outside_p1 = [s for s in range(4) if s not in X1]

print("Player 1 outside strategies:",
      [strategies[s] for s in outside_p1])

Player 1 outside strategies: ['D']


In [10]:
strategies = ["A", "B", "C", "D"]

X1 = [0, 1, 2]   # A, B, C
X2 = [0, 1]      # A, B

outside_p1 = [s for s in range(4) if s not in X1]

print("Player 1 outside strategies:",
      [strategies[s] for s in outside_p1])

def can_be_best_response_p1(target_strategy, X2, U1):

    # No optimization objective, only feasibility
    c = np.zeros(len(X2))

    # Belief probabilities must sum to 1
    A_eq = [np.ones(len(X2))]
    b_eq = [1]

    # Each probability is between 0 and 1
    bounds = [(0, 1)] * len(X2)

    # Best-response inequalities
    A_ub = []
    b_ub = []

    for other_strategy in range(4):

        if other_strategy == target_strategy:
            continue

        difference = (
            U1[other_strategy, X2]
            - U1[target_strategy, X2]
        )

        A_ub.append(difference)
        b_ub.append(0)

    result = linprog(
        c,
        A_ub=A_ub,
        b_ub=b_ub,
        A_eq=A_eq,
        b_eq=b_eq,
        bounds=bounds,
        method="highs"
    )

    return result.success

for s in outside_p1:
    print(
        strategies[s],
        can_be_best_response_p1(s, X2, U1)
    )

Player 1 outside strategies: ['D']
D True


In [11]:
def can_be_best_response_p2(target_strategy, X1, U2):

    # Belief over Player 1's strategies
    c = np.zeros(len(X1))

    # Probabilities sum to 1
    A_eq = [np.ones(len(X1))]
    b_eq = [1]

    # Each probability is between 0 and 1
    bounds = [(0, 1)] * len(X1)

    # Best-response inequalities
    A_ub = []
    b_ub = []

    for other_strategy in range(4):

        if other_strategy == target_strategy:
            continue

        difference = (
            U2[X1, other_strategy]
            - U2[X1, target_strategy]
        )

        A_ub.append(difference)
        b_ub.append(0)

    result = linprog(
        c,
        A_ub=A_ub,
        b_ub=b_ub,
        A_eq=A_eq,
        b_eq=b_eq,
        bounds=bounds,
        method="highs"
    )

    return result.success

In [13]:
def is_curb(X1, X2, U1, U2):

    # Check Player 1's outside strategies
    outside_p1 = [s for s in range(4) if s not in X1]

    for s in outside_p1:
        if can_be_best_response_p1(s, X2, U1):
            return False

    # Check Player 2's outside strategies
    outside_p2 = [s for s in range(4) if s not in X2]

    for s in outside_p2:
        if can_be_best_response_p2(s, X1, U2):
            return False

    return True

print(is_curb(X1, X2, U1, U2))

False


In [15]:
from itertools import combinations

def all_nonempty_subsets(n=4):
    subsets = []

    for r in range(1, n + 1):
        for combo in combinations(range(n), r):
            subsets.append(list(combo))

    return subsets

all_subsets = all_nonempty_subsets()

print("Number of subsets:", len(all_subsets))
print(all_subsets)

is_curb(X1, X2, U1, U2)

Number of subsets: 15
[[0], [1], [2], [3], [0, 1], [0, 2], [0, 3], [1, 2], [1, 3], [2, 3], [0, 1, 2], [0, 1, 3], [0, 2, 3], [1, 2, 3], [0, 1, 2, 3]]


False

In [17]:
curb_sets = []

for X1 in all_subsets:
    for X2 in all_subsets:
        if is_curb(X1, X2, U1, U2):
            curb_sets.append((X1, X2))

print("Number of CURB sets:", len(curb_sets))

curb_sets.append((X1, X2))

Number of CURB sets: 23


In [18]:
def names(X):
    return [strategies[i] for i in X]

for X1, X2 in curb_sets:
    print(names(X1), "x", names(X2))

['D'] x ['B']
['D'] x ['A', 'B']
['D'] x ['B', 'D']
['D'] x ['A', 'B', 'D']
['A', 'D'] x ['A', 'B']
['A', 'D'] x ['A', 'B', 'D']
['B', 'D'] x ['B']
['B', 'D'] x ['A', 'B']
['B', 'D'] x ['B', 'D']
['B', 'D'] x ['A', 'B', 'D']
['C', 'D'] x ['B']
['C', 'D'] x ['A', 'B']
['C', 'D'] x ['B', 'D']
['C', 'D'] x ['A', 'B', 'D']
['A', 'B', 'D'] x ['A', 'B', 'C']
['A', 'B', 'D'] x ['A', 'B', 'C', 'D']
['A', 'C', 'D'] x ['A', 'B']
['A', 'C', 'D'] x ['A', 'B', 'D']
['B', 'C', 'D'] x ['B']
['B', 'C', 'D'] x ['A', 'B']
['B', 'C', 'D'] x ['B', 'D']
['B', 'C', 'D'] x ['A', 'B', 'D']
['A', 'B', 'C', 'D'] x ['A', 'B', 'C', 'D']
['A', 'B', 'C', 'D'] x ['A', 'B', 'C', 'D']


In [19]:
def is_subset(Y, X):
    return set(Y).issubset(set(X))

In [20]:
minimal_curb_sets = []

for X1, X2 in curb_sets:
    is_minimal = True

    for Y1, Y2 in curb_sets:
        if (Y1, Y2) == (X1, X2):
            continue

        if is_subset(Y1, X1) and is_subset(Y2, X2):
            is_minimal = False
            break

    if is_minimal:
        minimal_curb_sets.append((X1, X2))

In [21]:
print("Number of minimal CURB sets:", len(minimal_curb_sets))

for X1, X2 in minimal_curb_sets:
    print(names(X1), "x", names(X2))

Number of minimal CURB sets: 1
['D'] x ['B']


In [23]:
import numpy as np
from scipy.optimize import linprog
from itertools import combinations

In [24]:
def all_nonempty_subsets(n=4):
    subsets = []

    for r in range(1, n + 1):
        for combo in combinations(range(n), r):
            subsets.append(list(combo))

    return subsets

In [25]:
def can_be_best_response_p1(target_strategy, X2, U1):

    c = np.zeros(len(X2))

    A_eq = [np.ones(len(X2))]
    b_eq = [1]

    bounds = [(0, 1)] * len(X2)

    A_ub = []
    b_ub = []

    for other_strategy in range(4):
        if other_strategy == target_strategy:
            continue

        difference = (
            U1[other_strategy, X2]
            - U1[target_strategy, X2]
        )

        A_ub.append(difference)
        b_ub.append(0)

    result = linprog(
        c,
        A_ub=A_ub,
        b_ub=b_ub,
        A_eq=A_eq,
        b_eq=b_eq,
        bounds=bounds,
        method="highs"
    )

    return result.success

In [26]:
def can_be_best_response_p2(target_strategy, X1, U2):

    c = np.zeros(len(X1))

    A_eq = [np.ones(len(X1))]
    b_eq = [1]

    bounds = [(0, 1)] * len(X1)

    A_ub = []
    b_ub = []

    for other_strategy in range(4):
        if other_strategy == target_strategy:
            continue

        difference = (
            U2[X1, other_strategy]
            - U2[X1, target_strategy]
        )

        A_ub.append(difference)
        b_ub.append(0)

    result = linprog(
        c,
        A_ub=A_ub,
        b_ub=b_ub,
        A_eq=A_eq,
        b_eq=b_eq,
        bounds=bounds,
        method="highs"
    )

    return result.success

In [27]:
def is_curb(X1, X2, U1, U2):

    outside_p1 = [s for s in range(4) if s not in X1]

    for s in outside_p1:
        if can_be_best_response_p1(s, X2, U1):
            return False

    outside_p2 = [s for s in range(4) if s not in X2]

    for s in outside_p2:
        if can_be_best_response_p2(s, X1, U2):
            return False

    return True

In [28]:
def find_minimal_curb_sets(U1, U2):

    all_subsets = all_nonempty_subsets(4)

    curb_sets = []

    for X1 in all_subsets:
        for X2 in all_subsets:
            if is_curb(X1, X2, U1, U2):
                curb_sets.append((X1, X2))

    minimal_curb_sets = []

    for X1, X2 in curb_sets:
        is_minimal = True

        for Y1, Y2 in curb_sets:

            if (Y1, Y2) == (X1, X2):
                continue

            if (
                set(Y1).issubset(set(X1))
                and set(Y2).issubset(set(X2))
            ):
                is_minimal = False
                break

        if is_minimal:
            minimal_curb_sets.append((X1, X2))

    return minimal_curb_sets